# Train Stockout Forecasting Models

Colab-ready training notebook for focused stockout architecture comparison: persistence, MLP, GRU, LSTM, Seq2Seq Transformer, TFT-style, and PatchTST-style.

In [ ]:
try:
    import pyarrow  # noqa: F401
except ImportError:
    !pip -q install pyarrow

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple
import math
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CKPT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
@dataclass
class Config:
    history_len: int = 28
    forecast_len: int = 7
    val_days: int = 15
    batch_size: int = 256
    epochs: int = 8
    lr: float = 1e-3
    weight_decay: float = 1e-4
    hidden_dim: int = 96
    embed_dim: int = 32
    num_layers: int = 2
    num_heads: int = 4
    dropout: float = 0.15
    patience: int = 3
    seed: int = 42
    use_pos_weight: bool = True

CFG = Config()

STATIC_CAT_COLS = ["city_id", "store_id", "management_group_id", "first_category_id", "second_category_id", "third_category_id", "product_id"]
TEMPORAL_CAT_COLS = ["holiday_flag", "activity_flag"]
CAT_COLS = STATIC_CAT_COLS + TEMPORAL_CAT_COLS
HIST_NUM_COLS = ["discount", "precpt", "avg_temperature", "avg_humidity", "avg_wind_level", "sale_amount", "stockout"]
FUTURE_NUM_COLS = ["discount", "precpt", "avg_temperature", "avg_humidity", "avg_wind_level"]
TARGET_COL = "stockout"

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Data Pipeline

In [ ]:
class CategoryEncoder:
    def __init__(self, cols: List[str]):
        self.cols = cols
        self.maps: Dict[str, Dict[str, int]] = {}
        self.sizes: Dict[str, int] = {}

    def fit(self, df: pd.DataFrame):
        for col in self.cols:
            values = df[col].fillna("__NA__").astype(str).unique().tolist()
            mapping = {"__UNK__": 0}
            mapping.update({v: i + 1 for i, v in enumerate(sorted(values))})
            self.maps[col] = mapping
            self.sizes[col] = len(mapping)
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        for col in self.cols:
            mapping = self.maps[col]
            out[col] = out[col].fillna("__NA__").astype(str).map(mapping).fillna(0).astype(np.int64)
        return out

def load_data():
    train = pd.read_parquet(DATA_DIR / "top15_train.parquet")
    test = pd.read_parquet(DATA_DIR / "top15_test.parquet")
    for df in (train, test):
        df["dt"] = pd.to_datetime(df["dt"])
        df[TARGET_COL] = (df["stock_hour6_22_cnt"] > 0).astype(np.float32)
    return train, test

def split_dates(train_df: pd.DataFrame, val_days: int):
    dates = np.array(sorted(train_df["dt"].unique()))
    val_start = pd.Timestamp(dates[-val_days])
    train_end = pd.Timestamp(dates[-val_days - 1])
    return train_end, val_start

def fit_apply_preprocessors(train_df: pd.DataFrame, test_df: pd.DataFrame, train_end: pd.Timestamp):
    fit_rows = train_df[train_df["dt"] <= train_end].copy()
    cat_encoder = CategoryEncoder(CAT_COLS).fit(fit_rows)
    scale_cols = sorted((set(HIST_NUM_COLS + FUTURE_NUM_COLS) - {TARGET_COL}))
    scaler = StandardScaler().fit(fit_rows[scale_cols].fillna(0.0))
    combined = pd.concat([train_df.assign(source="train"), test_df.assign(source="test")], ignore_index=True)
    combined = cat_encoder.transform(combined)
    combined[scale_cols] = scaler.transform(combined[scale_cols].fillna(0.0))
    return combined, cat_encoder

def make_windows(df: pd.DataFrame, cfg: Config, train_df: pd.DataFrame, train_end: pd.Timestamp, val_start: pd.Timestamp):
    rows = []
    train_max = train_df["dt"].max()
    df = df.sort_values(["store_id", "product_id", "dt"]).reset_index(drop=True)
    for _, group in df.groupby(["store_id", "product_id"], sort=False):
        group = group.sort_values("dt")
        if len(group) < cfg.history_len + cfg.forecast_len:
            continue
        hist_num = group[HIST_NUM_COLS].to_numpy(np.float32)
        fut_num = group[FUTURE_NUM_COLS].to_numpy(np.float32)
        cats = group[CAT_COLS].to_numpy(np.int64)
        target = group[TARGET_COL].to_numpy(np.float32)
        dates = group["dt"].to_numpy()
        for start in range(0, len(group) - cfg.history_len - cfg.forecast_len + 1):
            hist_end = start + cfg.history_len
            fut_end = hist_end + cfg.forecast_len
            target_start = pd.Timestamp(dates[hist_end])
            target_end = pd.Timestamp(dates[fut_end - 1])
            if target_end <= train_end:
                split = "train"
            elif target_start >= val_start and target_end <= train_max:
                split = "val"
            elif target_start > train_max:
                split = "test"
            else:
                continue
            rows.append({"hist_num": hist_num[start:hist_end], "hist_cat": cats[start:hist_end], "future_num": fut_num[hist_end:fut_end], "future_cat": cats[hist_end:fut_end], "target": target[hist_end:fut_end], "split": split})
    return rows

class StockoutSequenceDataset(Dataset):
    def __init__(self, windows, split: str):
        self.windows = [w for w in windows if w["split"] == split]
    def __len__(self):
        return len(self.windows)
    def __getitem__(self, idx):
        w = self.windows[idx]
        return {
            "hist_num": torch.tensor(w["hist_num"], dtype=torch.float32),
            "hist_cat": torch.tensor(w["hist_cat"], dtype=torch.long),
            "future_num": torch.tensor(w["future_num"], dtype=torch.float32),
            "future_cat": torch.tensor(w["future_cat"], dtype=torch.long),
            "target": torch.tensor(w["target"], dtype=torch.float32),
        }

train_df, test_df = load_data()
train_end, val_start = split_dates(train_df, CFG.val_days)
encoded, cat_encoder = fit_apply_preprocessors(train_df, test_df, train_end)
windows = make_windows(encoded, CFG, train_df, train_end, val_start)
datasets = {s: StockoutSequenceDataset(windows, s) for s in ["train", "val", "test"]}
loaders = {"train": DataLoader(datasets["train"], CFG.batch_size, shuffle=True), "val": DataLoader(datasets["val"], CFG.batch_size, shuffle=False), "test": DataLoader(datasets["test"], CFG.batch_size, shuffle=False)}
print({s: len(ds) for s, ds in datasets.items()})
batch = next(iter(loaders["train"]))
print({k: tuple(v.shape) for k, v in batch.items()})
assert batch["target"].shape[1] == CFG.forecast_len
assert not torch.isnan(batch["hist_num"]).any()
assert not torch.isnan(batch["future_num"]).any()

## Metrics And Baselines

In [ ]:
def flatten_np(values):
    return np.concatenate(values, axis=0).reshape(-1)

def metric_dict(y_true, y_prob, prefix=""):
    y_true = np.asarray(y_true).reshape(-1)
    y_prob = np.asarray(y_prob).reshape(-1)
    y_pred = (y_prob >= 0.5).astype(int)
    out = {}
    if len(np.unique(y_true)) > 1:
        out[prefix + "roc_auc"] = roc_auc_score(y_true, y_prob)
        out[prefix + "pr_auc"] = average_precision_score(y_true, y_prob)
    else:
        out[prefix + "roc_auc"] = np.nan
        out[prefix + "pr_auc"] = np.nan
    out[prefix + "accuracy"] = accuracy_score(y_true, y_pred)
    out[prefix + "f1"] = f1_score(y_true, y_pred, zero_division=0)
    out[prefix + "precision"] = precision_score(y_true, y_pred, zero_division=0)
    out[prefix + "recall"] = recall_score(y_true, y_pred, zero_division=0)
    out[prefix + "brier"] = brier_score_loss(y_true, y_prob)
    return out

def evaluate_probs(y_true_2d, y_prob_2d):
    metrics = metric_dict(y_true_2d.reshape(-1), y_prob_2d.reshape(-1))
    for h in range(y_true_2d.shape[1]):
        metrics.update(metric_dict(y_true_2d[:, h], y_prob_2d[:, h], prefix=f"h{h+1}_"))
    return metrics

def persistence_predict(loader):
    probs, targets = [], []
    for batch in loader:
        recent = batch["hist_num"][:, -7:, HIST_NUM_COLS.index("stockout")].mean(dim=1, keepdim=True)
        probs.append(recent.repeat(1, CFG.forecast_len).numpy())
        targets.append(batch["target"].numpy())
    return np.concatenate(targets), np.concatenate(probs)

baseline_rows = []
for split in ["val", "test"]:
    y, p = persistence_predict(loaders[split])
    row = {"model": "persistence", "split": split}
    row.update(evaluate_probs(y, p))
    baseline_rows.append(row)
pd.DataFrame(baseline_rows)

## Model Definitions

In [ ]:
class CatEmbedder(nn.Module):
    def __init__(self, cat_sizes: List[int], embed_dim: int):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(size, embed_dim) for size in cat_sizes])
        self.out_dim = len(cat_sizes) * embed_dim
    def forward(self, x_cat):
        return torch.cat([emb(x_cat[..., i]) for i, emb in enumerate(self.embeddings)], dim=-1)

class MLPBaseline(nn.Module):
    def __init__(self, hist_num_dim, cat_sizes, cfg: Config):
        super().__init__()
        self.cat_embed = CatEmbedder(cat_sizes, cfg.embed_dim)
        in_dim = cfg.history_len * hist_num_dim + self.cat_embed.out_dim
        self.net = nn.Sequential(nn.Linear(in_dim, cfg.hidden_dim), nn.ReLU(), nn.Dropout(cfg.dropout), nn.Linear(cfg.hidden_dim, cfg.hidden_dim), nn.ReLU(), nn.Linear(cfg.hidden_dim, cfg.forecast_len))
    def forward(self, batch):
        hist = batch["hist_num"].flatten(1)
        static = self.cat_embed(batch["hist_cat"][:, -1, :])
        return self.net(torch.cat([hist, static], dim=-1))

class RNNSeq2Seq(nn.Module):
    def __init__(self, hist_num_dim, future_num_dim, cat_sizes, cfg: Config, cell="gru"):
        super().__init__()
        self.cat_embed = CatEmbedder(cat_sizes, cfg.embed_dim)
        rnn_cls = nn.GRU if cell == "gru" else nn.LSTM
        hist_in = hist_num_dim + self.cat_embed.out_dim
        fut_in = future_num_dim + self.cat_embed.out_dim
        self.encoder = rnn_cls(hist_in, cfg.hidden_dim, cfg.num_layers, batch_first=True, dropout=cfg.dropout if cfg.num_layers > 1 else 0)
        self.decoder = rnn_cls(fut_in, cfg.hidden_dim, cfg.num_layers, batch_first=True, dropout=cfg.dropout if cfg.num_layers > 1 else 0)
        self.head = nn.Linear(cfg.hidden_dim, 1)
    def forward(self, batch):
        hcat = self.cat_embed(batch["hist_cat"])
        fcat = self.cat_embed(batch["future_cat"])
        enc_in = torch.cat([batch["hist_num"], hcat], dim=-1)
        dec_in = torch.cat([batch["future_num"], fcat], dim=-1)
        _, state = self.encoder(enc_in)
        out, _ = self.decoder(dec_in, state)
        return self.head(out).squeeze(-1)

class Seq2SeqTransformer(nn.Module):
    def __init__(self, hist_num_dim, future_num_dim, cat_sizes, cfg: Config):
        super().__init__()
        self.cat_embed = CatEmbedder(cat_sizes, cfg.embed_dim)
        self.hist_proj = nn.Linear(hist_num_dim + self.cat_embed.out_dim, cfg.hidden_dim)
        self.fut_proj = nn.Linear(future_num_dim + self.cat_embed.out_dim, cfg.hidden_dim)
        self.hist_pos = nn.Parameter(torch.randn(1, cfg.history_len, cfg.hidden_dim) * 0.02)
        self.fut_pos = nn.Parameter(torch.randn(1, cfg.forecast_len, cfg.hidden_dim) * 0.02)
        self.transformer = nn.Transformer(d_model=cfg.hidden_dim, nhead=cfg.num_heads, num_encoder_layers=cfg.num_layers, num_decoder_layers=cfg.num_layers, dropout=cfg.dropout, batch_first=True)
        self.head = nn.Linear(cfg.hidden_dim, 1)
    def forward(self, batch):
        src = self.hist_proj(torch.cat([batch["hist_num"], self.cat_embed(batch["hist_cat"])], dim=-1)) + self.hist_pos
        tgt = self.fut_proj(torch.cat([batch["future_num"], self.cat_embed(batch["future_cat"])], dim=-1)) + self.fut_pos
        out = self.transformer(src=src, tgt=tgt)
        return self.head(out).squeeze(-1)

class TFTStyleModel(nn.Module):
    def __init__(self, hist_num_dim, future_num_dim, cat_sizes, cfg: Config):
        super().__init__()
        self.cat_embed = CatEmbedder(cat_sizes, cfg.embed_dim)
        self.hist_proj = nn.Linear(hist_num_dim + self.cat_embed.out_dim, cfg.hidden_dim)
        self.future_proj = nn.Linear(future_num_dim + self.cat_embed.out_dim, cfg.hidden_dim)
        self.static_gate = nn.Sequential(nn.Linear(self.cat_embed.out_dim, cfg.hidden_dim), nn.Sigmoid())
        self.encoder = nn.LSTM(cfg.hidden_dim, cfg.hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(cfg.hidden_dim, cfg.hidden_dim, batch_first=True)
        layer = nn.TransformerEncoderLayer(cfg.hidden_dim, cfg.num_heads, cfg.hidden_dim * 2, cfg.dropout, batch_first=True)
        self.attn = nn.TransformerEncoder(layer, num_layers=1)
        self.head = nn.Linear(cfg.hidden_dim, 1)
    def forward(self, batch):
        hcat = self.cat_embed(batch["hist_cat"])
        fcat = self.cat_embed(batch["future_cat"])
        static = self.cat_embed(batch["hist_cat"][:, -1, :])
        gate = self.static_gate(static).unsqueeze(1)
        enc_in = self.hist_proj(torch.cat([batch["hist_num"], hcat], dim=-1)) * gate
        dec_in = self.future_proj(torch.cat([batch["future_num"], fcat], dim=-1)) * gate
        _, state = self.encoder(enc_in)
        dec, _ = self.decoder(dec_in, state)
        dec = self.attn(dec)
        return self.head(dec).squeeze(-1)

class PatchTSTStyleModel(nn.Module):
    def __init__(self, hist_num_dim, cat_sizes, cfg: Config, patch_len=7):
        super().__init__()
        self.patch_len = patch_len
        self.num_patches = cfg.history_len // patch_len
        self.cat_embed = CatEmbedder(cat_sizes, cfg.embed_dim)
        self.patch_proj = nn.Linear(hist_num_dim * patch_len, cfg.hidden_dim)
        self.static_proj = nn.Linear(self.cat_embed.out_dim, cfg.hidden_dim)
        self.pos = nn.Parameter(torch.randn(1, self.num_patches, cfg.hidden_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(cfg.hidden_dim, cfg.num_heads, cfg.hidden_dim * 2, cfg.dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, cfg.num_layers)
        self.head = nn.Linear(cfg.hidden_dim, cfg.forecast_len)
    def forward(self, batch):
        x = batch["hist_num"][:, -self.num_patches * self.patch_len:, :]
        b, _, f = x.shape
        patches = x.reshape(b, self.num_patches, self.patch_len * f)
        static = self.static_proj(self.cat_embed(batch["hist_cat"][:, -1, :])).unsqueeze(1)
        z = self.patch_proj(patches) + self.pos + static
        z = self.encoder(z).mean(dim=1)
        return self.head(z)

cat_sizes = [cat_encoder.sizes[c] for c in CAT_COLS]
hist_num_dim = len(HIST_NUM_COLS)
future_num_dim = len(FUTURE_NUM_COLS)

model_factories = {
    "mlp_lag": lambda: MLPBaseline(hist_num_dim, cat_sizes, CFG),
    "gru_seq2seq": lambda: RNNSeq2Seq(hist_num_dim, future_num_dim, cat_sizes, CFG, cell="gru"),
    "lstm_seq2seq": lambda: RNNSeq2Seq(hist_num_dim, future_num_dim, cat_sizes, CFG, cell="lstm"),
    "seq2seq_transformer": lambda: Seq2SeqTransformer(hist_num_dim, future_num_dim, cat_sizes, CFG),
    "tft_style": lambda: TFTStyleModel(hist_num_dim, future_num_dim, cat_sizes, CFG),
    "patchtst_style": lambda: PatchTSTStyleModel(hist_num_dim, cat_sizes, CFG),
}

for name, factory in model_factories.items():
    m = factory().to(device)
    with torch.no_grad():
        out = m({k: v.to(device) for k, v in batch.items()})
    print(name, tuple(out.shape))
    assert out.shape == batch["target"].shape

## Training Loop

In [ ]:
def to_device(batch):
    return {k: v.to(device) for k, v in batch.items()}

def collect_predictions(model, loader):
    model.eval()
    probs, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = to_device(batch)
            logits = model(batch)
            probs.append(torch.sigmoid(logits).cpu().numpy())
            targets.append(batch["target"].cpu().numpy())
    return np.concatenate(targets), np.concatenate(probs)

def train_one_model(name, model):
    model = model.to(device)
    train_targets = np.concatenate([w["target"] for w in datasets["train"].windows])
    pos = train_targets.sum()
    neg = train_targets.size - pos
    pos_weight = torch.tensor([neg / max(pos, 1)], device=device) if CFG.use_pos_weight else None
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    best_pr_auc = -np.inf
    best_path = CKPT_DIR / f"{name}.pt"
    stale = 0

    for epoch in range(1, CFG.epochs + 1):
        model.train()
        losses = []
        pbar = tqdm(loaders["train"], desc=f"{name} epoch {epoch}", leave=False)
        for batch in pbar:
            batch = to_device(batch)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(batch)
                loss = criterion(logits, batch["target"])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())
            pbar.set_postfix(loss=np.mean(losses))

        y_val, p_val = collect_predictions(model, loaders["val"])
        metrics = evaluate_probs(y_val, p_val)
        print(f"{name} epoch {epoch}: loss={np.mean(losses):.4f} val_pr_auc={metrics['pr_auc']:.4f} val_roc_auc={metrics['roc_auc']:.4f}")
        if metrics["pr_auc"] > best_pr_auc:
            best_pr_auc = metrics["pr_auc"]
            stale = 0
            torch.save({"model_state": model.state_dict(), "config": CFG.__dict__, "cat_sizes": cat_sizes}, best_path)
        else:
            stale += 1
            if stale >= CFG.patience:
                break

    model.load_state_dict(torch.load(best_path, map_location=device)["model_state"])
    rows = []
    for split in ["val", "test"]:
        y, p = collect_predictions(model, loaders[split])
        row = {"model": name, "split": split}
        row.update(evaluate_probs(y, p))
        rows.append(row)
        pd.DataFrame({f"prob_h{i+1}": p[:, i] for i in range(CFG.forecast_len)} | {f"target_h{i+1}": y[:, i] for i in range(CFG.forecast_len)}).to_csv(OUTPUT_DIR / f"{name}_{split}_predictions.csv", index=False)
    return rows

In [ ]:
# Choose a subset during debugging, then run all models on Colab.
RUN_MODELS = [
    "mlp_lag",
    "gru_seq2seq",
    "lstm_seq2seq",
    "seq2seq_transformer",
    "tft_style",
    "patchtst_style",
]

all_rows = list(baseline_rows)
for model_name in RUN_MODELS:
    seed_everything(CFG.seed)
    all_rows.extend(train_one_model(model_name, model_factories[model_name]()))

metrics_df = pd.DataFrame(all_rows).sort_values(["split", "pr_auc"], ascending=[True, False])
metrics_df.to_csv(OUTPUT_DIR / "metrics_summary.csv", index=False)
display(metrics_df[["model", "split", "pr_auc", "roc_auc", "f1", "precision", "recall", "accuracy", "brier"]])